# Anuario de Aforos: seleccionar estaciones
***

***Autor:** Jesús Casado Rodríguez*<br>
***Fecha:** 23-03-2026*<br>

**Introducción:**<br>
En este _notebook_ se analizan los datos del *Anuario de Aforos* y se hace una selección de las estaciones a incluir en CAMELS-ES en base a:

1. Disponibilidad mínima de datos en la serie de caudal.
2. Series de caudal con mínima alteración del régimen natural, puesto que las series claramente influidas por embalses no son propicias para el modelado.
3. Eliminar estaciones que por cercanía están altamente correlacionadas.

**Por hacer**:


In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
# from datetime import datetime
from tqdm.notebook import tqdm
from pathlib import Path
import yaml

from camels_es.config import Config
from camels_es.anuario_aforos import extraer_estaciones, extraer_caudal_estaciones, extraer_embalses, periodo_estudio
from camels_es.plots import plot_stations, plot_discharge, create_station_html#, create_index
from camels_es.catchment.utils import lfcoords_csv

In [ ]:
import unicodedata
import re

def slugify(value):
    # Remove accents
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    # Remove non-alphanumeric characters and replace spaces with hyphens
    value = re.sub(r'[^\w\s-]', '', value).strip().lower()
    return re.sub(r'[-\s]+', '-', value)

## Configuración

In [ ]:
cfg = Config('../config_CAMELS.yml')

## Estaciones

### Anuario de Aforos

In [ ]:
# identificar demarcaciones hidrográficas
cuencas = {dir.stem: dir for dir in cfg.path_in.iterdir() if dir.is_dir() &  (dir.stem not in ['estaciones', 'GIS', 'timeseries'])}

# extraer estaciones del Anuario de Aforos
if 'estaciones' in locals():
    del estaciones
for cuenca, folder in cuencas.items():
    stns = extraer_estaciones(
        folder / 'estaf.csv',
        min_area=cfg.area_min, 
        max_area=cfg.area_max, 
        years=cfg.min_años, 
        epsg=cfg.crs)
    stns['cuenca'] = cuenca
    if 'estaciones' in locals():
        estaciones = pd.concat((estaciones, stns))
    else:
        estaciones = stns
    print('nº de estaciones en el {0:>12}:\t{1:>4}'.format(cuenca.capitalize(), stns.shape[0]))
estaciones.sort_index(axis=0, inplace=True)
# estaciones.index = estaciones.index.astype(int)

# mapa de las estaciones
plot_stations(
    estaciones.geometry, 
    area=estaciones.suprest, 
    extent=[-9.5, 3.5, 36, 44.5],
    title=f'Estaciones en el Anuario de Aforos',
    save=cfg.path_plots / 'estaciones.jpg'
)

print('\nnº total de estaciones:\t\t\t{0:>4}'.format(estaciones.shape[0]))
print('nº de estaciones en servicio:\t\t{0:>4}'.format(estaciones.loc[estaciones.serv == 1].shape[0]))

### MITECO

In [ ]:
# cargar tabla de atributos de las estaciones
est_miteco = pd.read_csv(
    cfg.path_in / 'estaciones' / 'Situac_4_Rio_rev.csv',
    encoding='latin1',
    index_col='COD_HIDRO'
    )
est_miteco.columns = est_miteco.columns.str.lower()
est_miteco.rename(
    columns={'regimen_rio': 'regimen', 'sistema_explo': 'sistema'}, 
    inplace=True
    )
# print('Nº estaciones:\t{0}\nNº atributos:\t{1}'.format(*est_miteco.shape))

# adaptar campo "regimen"
cols = ['rio', 'regimen', 'sistema']
for col in cols:
    est_miteco[col] = est_miteco[col].str.lower()
    est_miteco.loc[est_miteco[col].isnull(), col] = pd.NA
est_miteco.regimen = est_miteco.regimen.replace({
    'muy modificado': 'alterado',
    '-': 'desconocido',
    pd.NA: 'desconocido'
    })

# añadir campos 'rio' y 'regimen' a estaciones
ids = estaciones.index.intersection(est_miteco.index)
estaciones.loc[ids, cols] = est_miteco.loc[ids, cols]

del est_miteco

print('\nnº total de estaciones:\t\t\t{0:>4}'.format(estaciones.shape[0]))

## Embalses

In [ ]:
file_reservoirs = cfg.path_GIS / 'embalses.geojson'
file_mapping = cfg.path_GIS / 'embalses_codigos.yml'
if file_reservoirs.is_file():
    print(f'Leyendo embalses de: {file_reservoirs}')

    # cargar capa de embalses
    embalses = gpd.read_file(file_reservoirs).set_index('ref_ceh')
    embalses.index = embalses.index.astype(int)

    # cargar correspondencia de códigos de embalse
    if file_mapping.is_file():
        # leer archivo
        with open(file_mapping, 'r') as file:
            mapping_reservoirs = yaml.safe_load(file)
        # añadir códigos
        cols = ['SNCZI', 'GDW_ID']
        for col in cols:
            embalses[col] = embalses.index.map({ID: dct[col] for ID, dct in mapping_reservoirs.items()})
        embalses[cols] = embalses[cols].astype('Int64')
else:
    # extraer embalses del Anuario de Aforos
    embalses = []
    for cuenca in tqdm(cuencas):
        res = extraer_embalses(cfg.path_in / cuenca / 'embalse.csv', epsg=cfg.crs)
        res['cuenca'] = cuenca
        embalses.append(res)
        print('nº de embalses en el {0:>12}:\t{1:>3}'.format(cuenca.capitalize(), res.shape[0]))
        del res
    embalses = pd.concat(embalses, axis=0)
    
    # Export
    embalses.to_file(file_reservoirs, driver='GeoJSON')
    print(f'Los embalses se han guardado en: {file_reservoirs}')

    # # create mapping between reservoir codes
    # cols = ['SNCZI', 'GDW_ID']
    # reservoirs_AA[cols] = reservoirs_AA[cols].astype('Int64')
    # reservoirs_AA[cols] = reservoirs_AA[cols].replace(pd.NA, None)
    # mapping_reservoirs = {}
    # for ID, attrs in reservoirs_AA.iterrows():
    #     mapping_reservoirs[ID] = {col: attrs[col] if pd.notnull(attrs[col]) else None for col in cols}

    # # save mapping
    # with open(cfg.path_GIS / 'embalses_codigos.yml', 'w') as file:
    #     yaml.dump(mapping_reservoirs, file)#, default_flow_style=False)#, sort_keys=True)

# mapa de los embalses
plot_stations(
    embalses.geometry, 
    # area=estaciones.suprest, 
    extent=[-9.5, 3.5, 36, 44.5],
    marker='v',
    color='k',
    title=f'Embalses en el Anuario de Aforos',
    # save=cfg.path_plots / 'embalses.jpg'
)

print('\nnº total de embalses:\t\t\t{0:>3}'.format(embalses.shape[0]))
print('nº de embalses en servicio:\t\t{0:>3}'.format(embalses.loc[embalses.serv == 1].shape[0]))

In [ ]:
cfg.path_GIS

In [ ]:
path_beaver = Path('~/Data/BEAVER-ES/GIS')
embalses.to_file(path_beaver / 'beaverses.geojson', driver='GeoJSON')

In [ ]:
embalses.GDW_ID.isnull().sum()

In [ ]:
egis_embalse = gpd.read_file('/mnt/c/Datasets/CEDEX/processed/reservoirs/attributes/GIS/egis_embalse.shp').set_index('CODIGO')
egis_embalse.index.name = 'SNCZI'

len(egis_embalse)

In [ ]:
egis_embalse.SUP_CUENCA.notnull().sum()

#### `lfcoords`

In [ ]:
snczi_ids = egis_embalse.index.intersection(embalses.SNCZI)
df = embalses.loc[embalses.SNCZI.isin(snczi_ids), ['geometry']]
df.index.name = 'ID'
df['lat'] = df.geometry.y
df['lon'] = df.geometry.x
# df['area'] = egis_embalse.loc[ids, 'SUP_CUENCA'].values

In [ ]:
egis_embalse.loc[snczi_ids, 'SUP_CUENCA'].isnull().sum()

In [ ]:
embalses.index.intersection(egis_embalse.index)

In [ ]:
egis_embalse.columns

In [ ]:
pd.read_csv('/mnt/c/Datasets/CEDEX/processed/reservoirs/attributes/attributes_reservoirs.csv', index_col='ref_ceh')

### `lfcoords`

In [ ]:
dams = pd.read_csv('/mnt/c/Datasets/CEDEX/processed/reservoirs/attributes/attributes_dams.csv', index_col='ref_ceh')
geometry = gpd.points_from_xy(dams['X-UTM30ETRS89'], dams['Y-UTM30ETRS89'], crs=25830).to_crs(4326)
dams = gpd.GeoDataFrame(dams, geometry=geometry)
# columns = {'X-UTM30ETRS89': 'X', 'Y-UTM30ETRS89': 'Y', 'Superficie de la cuenca hidrográfica (km2)': 'area'}
# dams.rename(columns=columns, inplace=True)

In [ ]:
csv_file = f'/home/casadoj/Data/BEAVER-ES/preprocessing/lfcoords/dams.csv'
area = dams['Superficie de la cuenca hidrográfica (km2)']
mask = area.notnull() & (area >= cfg.area_min)
lfcoords_csv(
    geometry=dams[mask].geometry,
    area=area[mask].round(2).astype(int),
    file=csv_file
)

In [ ]:
dams[mask]['Superficie de la cuenca hidrográfica (km2)'].min()

In [ ]:
df.drop('geometry').to_csv(/)

In [ ]:
df.head()

In [ ]:
dams['Superficie de la cuenca hidrográfica (km2)'].notnull().sum()

In [ ]:

dams[columns.values()]

In [ ]:
gpd.GeoDataFrame(
    dams,
    geometry=gpd.points_from_xy(dams.X, dams.Y, crs=25830)
)

In [ ]:
reservoirs_AA = gpd.read_file('/mnt/c/Datasets/CEDEX/processed/reservoirs/attributes/GIS/reservoirs.shp').set_index('ref_ceh')
reservoirs_AA.index = reservoirs_AA.index.astype(int)

reservoirs_AA.head()

In [ ]:
len(embalses), len(reservoirs_AA)

In [ ]:
embalses.index.difference(reservoirs_AA.index)

In [ ]:
embalses['CODIGO'] = reservoirs_AA['SNCZI']

In [ ]:
embalses[embalses.CODIGO.isnull()]

## Caudal 

In [ ]:
# extraer series de caudal del Anuario de Aforos
if 'caudal' in locals():
    del caudal
for cuenca in tqdm(cuencas):
    ids = estaciones[estaciones.cuenca == cuenca].index.to_list()
    q = extraer_caudal_estaciones(
        cfg.path_in / cuenca / 'afliq.csv', 
        indroea=ids, 
        start=cfg.start, 
        end=cfg.end
    )
    if 'caudal' in locals():
        caudal = pd.concat((caudal, q), axis=1)
    else:
        caudal = q
    print('nº de estaciones en las series de caudal del {0:>12}:\t{1:>3} ({2:>4})'.format(cuenca.capitalize(),
                                                                                  q.shape[1],
                                                                                  len(ids)))
caudal.sort_index(axis=1, inplace=True)
caudal.columns = caudal.columns.astype(int)

print('\nnº total de estaciones con serie de caudal:\t\t\t{0:>3} ({1:>4})'.format(caudal.shape[1], len(estaciones)))

In [ ]:
# gráfico de la disponibilidad de estaciones
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(caudal.shape[1] - caudal.isnull().sum(axis=1), lw=1)
ax.set(
    ylim=(-2, caudal.shape[1] + 2),
    xlim=(cfg.start, cfg.end), 
    ylabel='nº estaciones'
)
ax.axhline(estaciones.shape[0], c='k', lw=.5)
ax.set_title('disponibilidad de estaciones');

plt.savefig(cfg.path_plots / 'disponibilidad.jpg', dpi=300, bbox_inches='tight');

In [ ]:
# añadir un campo booleano en 'estaciones' si tiene o no datos de caudal
estaciones['caudal'] = 0
estaciones.loc[caudal.columns, 'caudal'] = 1
print('nº de estaciones con datos de caudal:\t\t{0}'.format(estaciones.caudal.sum()))

# definir el mejor periodo de estudio de cada estación
estaciones[['inicio', 'fin']] = periodo_estudio(caudal, cfg.disponibilidad, cfg.min_años)

# actualizar nº de años de datos diarios
estaciones['naa'] = (estaciones.fin - estaciones.inicio).astype('Int64')
mask_años = estaciones.naa >= cfg.min_años
print('nº de estaciones con {0} años de caudal:\t\t{1}'.format(cfg.min_años, mask_años.sum()))

#### `lfcoords`

In [ ]:
mask = mask_años & estaciones.caudal == 1
csv_file = f'/home/casadoj/Data/CAMELS-ES/preprocessing/lfcoords/stations_camelses_{mask.sum()}.csv'
lfcoords_csv(
    geometry=estaciones[mask].geometry,
    area= estaciones[mask].suprest,
    file=csv_file
)

### Hidrogramas

In [ ]:
regimenes = {
    'desconocido': 'dimgrey',
    'natural': 'steelblue',
    'alterado': 'indianred',
    'regulado': 'gold',
}

estaciones_sel = []
index_glob = []
estaciones['url_link'] = ''
path_ghpages = Path(os.getcwd()).parent.parent / 'docs'
url_ghpages = 'https://casadoj.github.io/CAMELS-ES'
for cuenca in tqdm(cuencas, desc='Cuencas'):

    # crear estructura de archivos
    cuenca_str = slugify(cuenca)
    path_basin = path_ghpages / cuenca_str
    index_glob.append((f'./{cuenca_str}/index.html', cuenca.capitalize()))
    index_basin = []

    # filtrar estaciones en la cuenca con datos de caudal
    stns_basin = estaciones.loc[(estaciones.cuenca == cuenca) & (estaciones.caudal == 1) & mask_años]

    # iterar por sistemas de explotación
    sistemas = list(set(stns_basin.sistema))
    pbar_sys = tqdm(sorted(sistemas), total=len(sistemas), desc='Sistema', leave=False)
    for sistema in pbar_sys:
        # índice y ruta del sistema
        sistema_str = slugify(sistema)
        path_sys = path_basin / sistema_str
        path_plots = path_sys / 'plots'
        path_plots.mkdir(parents=True, exist_ok=True)
        index_sys = []

        # actualizar índice de la demarcación
        index_basin.append((f'./{sistema_str}/index.html', sistema.capitalize()))
        
        # crear los hidrogramas
        stns = stns_basin.loc[stns_basin.sistema == sistema]
        stns.sort_index(inplace=True)
        pbar = tqdm(enumerate(stns.iterrows()), total=len(stns), desc='Estaciones', leave=False)
        for i, (id, attrs) in pbar:
            if id not in caudal.columns:
                continue

            # crear el hidrograma en HTML
            ts = caudal[id]
            ts = ts.loc[ts.first_valid_index():ts.last_valid_index()]
            title = '{0} - {1} - River {2} ({3})'.format(id, attrs.lugar.title(), attrs.rio.title(), cuenca.title())
            file = f'./plots/{id}.html'
            fig = plot_discharge(
                ts,
                area=attrs.suprest,
                title=title,
                regime=attrs.regimen,
                c=regimenes[attrs.regimen],
                save=True
            )
            create_station_html(
                fig, 
                path=path_sys / file, 
                start=ts.index.min().strftime('%Y-%m-%d'), 
                end=ts.index.max().strftime('%Y-%m-%d')
            )
            # create_station_html(
            #     ts, 
            #     area=attrs.suprest,
            #     title=title,
            #     regime=attrs.regimen,
            #     c=regimenes[attrs.regimen],
            #     save=path_sys / file,
            # )

            # guardar el archivo en el índice y el GeoDataFrame
            index_sys.append((file, title))
            stns_basin.loc[id, 'url_link'] = f'{url_ghpages}/{cuenca.title()}/{sistema.title()}/plots/{id}.html'

        # crear índice del sistema de explotación
        # create_index(
        #     path_sys, 
        #     title=sistema.lower(), 
        #     header=sistema.title(), 
        #     index=index_sys,
        #     target='_self',
        #     parent='Sistemas'
        #     )

    # crear índice de la cuenca
    # create_index(
    #     path_basin, 
    #     title=cuenca.lower(), 
    #     header=cuenca.title(), 
    #     index=index_basin, 
    #     target='_self',
    #     parent='Cuencas')

    estaciones_sel.append(stns_basin)

In [ ]:
# guardar shapefile
if isinstance(estaciones_sel, list):
    estaciones_sel = pd.concat(estaciones_sel, axis=0)
    for col in ['rio', 'sistema', 'cuenca']:
        estaciones[col] = estaciones[col].str.title()
# estaciones_sel.to_file(cfg.path_GIS / f'estaciones_caudal.shp')
estaciones_sel.to_file(path_ghpages / "estaciones.geojson", driver='GeoJSON')

# # crear índice global
# create_index(path_ghpages, title='camels-es', header='Cuencas', index=index_glob, target='_self')

In [ ]:
estaciones.loc[4267]

In [ ]:
import geopandas as gpd

In [ ]:
foo = gpd.read_file('/home/casadoj/Data/CAMELS-ES/preprocessing/lfcoords/stations_camelses_3sec.shp').set_index('ID')

In [ ]:
foo[foo.geometry.duplicated(keep=False)]

In [ ]:
estaciones[estaciones.suprest > estaciones.suprcnc]

## Selección de estaciones

***

In [ ]:
# estaciones a eliminar porque la serie está alterada por embalses
# las estaciones comentadas son etaciones dudosas
eliminar = {'CANTABRICO': ['1164','1175', '1186', '1196', '1207', '1215', '1237',
                           '1264', '1274', '1276', '1294', '1335', '1358', '1359',
                           '1425', '1446', '1588', '1805'],
            'DUERO': ['2001', '2002', '2004', '2010', '2011', '2018', '2019', '2020',
                      '2021', '2023', '2024', '2032', '2040', '2042', '2044', '2048', '2052', '2066',
                      '2073', '2075', '2078', '2084', '2087', '2088', '2091', '2094',
                      '2099', '2102', '2103', '2108', '2109', '2111', '2112', '2124', '2129',
                      '2133', '2134', '2135', '2137', '2140', '2148', '2149', '2161',
                      '2162', '2718', '2719'],
            #'2041', '2050', '2061', '2062', '2074'
            'EBRO': ['9001', '9005', '9007', '9008', '9010', '9012', '9013', '9014',
                     '9015', '9017', '9022', '9024', '9025', '9026', '9027', '9031',
                     '9032', '9034', '9035', '9036', '9042', '9047', '9048', '9049',
                     '9055', '9056', '9057', '9058', '9059', '9060', '9065', '9066', '9076', '9083',
                     '9084', '9087', '9094', '9096', '9097', '9099', '9100', '9101', '9105',
                     '9106', '9111', '9115', '9118', '9122', '9123', '9124', '9125',
                     '9126', '9127', '9137', '9142', '9145', '9147', '9155', '9161',
                     '9163', '9168', '9174', '9176', '9185', '9186', '9190', '9191', '9192',
                     '9193', '9201', '9209', '9216', '9225', '9229', '9230', '9231', '9250', '9250',
                     '9251', '9255', '9258', '9260', '9264', '9266', '9273', '9277', '9278', '9290',
                     '9292', '9307', '9315', '9319'],
            #, '9153', '9291', '9317'
            'GALICIA': ['1455', '1514', '1519', '1550', '1564'],
            'GUADALQUIVIR': ['5001', '5004', '5014', '5019', '5020', '5024', '5025',
                             '5027', '5041', '5042', '5048', '5077', '5081', '5082', '5084', '5090',
                             '5095', '5127', '5140', '5147', '5144', '5150'],
             # '5045', '5050', '5138', '5140', ''
            'GUADIANA': ['4004', '4009', '4013', '4030', '4105', '4201', '4202', '4203', '4207', '4209',
                         '4212', '4214', '4904'],
            # , ''
            'JUCAR': ['8005', '8015', '8022', '8025', '8027', '8032', '8036', '8042', '8071', '8074', '8089',
                      '8092', '8093', '8096', '8107', '8112', '8119', '8129', '8130', '8137', '8138', '8139',
                      '8140', '8144', '8145', '8147', '8148', '8153'],
                      #, '8028', '8032', '8060', '8104', '8120'
            'MINHO': ['1631', '1639', '1640', '1642', '1719', '1831'],
            'SEGURA': ['7001', '7003', '7004', '7006', '7013', '7016', '7018', '7029', '7030', '7055',
                       '7057', '7062', '7063', '7064', '7112', '7121', '7124', '7137', '7164', '7167', '7628'],
            # , '7112', '7117', '7121', '7165', ''
            'TAJO': ['3003', '3031', '3041', '3048', '3054', '3060', '3061', '3063', '3067', '3070', '3080',
                     '3082', '3147', '3149', '3153', '3158', '3162', '3164', '3172', '3173', '3177', '3183',
                     '3187', '3188', '3230', '3232', '3233', '3237', '3238', '3240', '3243', '3248',
                     '3251', '3254', '3255', '3258', '3259', '3270', '3271', '3273', '3281', '3904',
                      '3940'],
            #'3012', '3014', '3062, '3169', '3174', '3175', '3220', '3233', 
            #'3250', '3251', '3253', '3255', '3256', '3268', '3276', '3278', '3279', ''
            }

In [ ]:
# crear campo booleano con las estaciones seleccionadas
estaciones['sel'] = (~estaciones.inicio.isnull()).astype(int)
for cuenca in cuencas:
    estaciones.loc[eliminar[cuenca], 'sel'] = 0

print('nº de estaciones seleccionadas:\t{0}'.format((estaciones.sel == 1).sum()))

In [ ]:
# estaciones a eliminar porque hay otra estación cercana
eliminar2 = {'CANTABRICO': ['1303'],
            'DUERO': ['2076', '2070', '2068', '2117', '2710', '2123', '2097', '2085', ],
            'EBRO': ['9153', '9046', '9091', '9043', '9268', '9018', '9313', '9253', '9329', '9050'],
            'GALICIA': ['1440', '1483'],
            'GUADALQUIVIR': ['5076', '5128', '5080'],
            'GUADIANA': [],
            'JUCAR': ['8030'],
            'MINHO': ['1644', '1645','1626', '1625', '1628', '1621', '1607', '1608', '1617', '1754',
                      '1727', '1722', '1724'],
            'SEGURA': ['7165', '7117', '7129'],
            'TAJO': ['3194', '3102', '3159'],
            }

In [ ]:
for cuenca in cuencas:
    estaciones.loc[eliminar2[cuenca], 'sel'] = 0

print('nº de estaciones seleccionadas:\t{0}'.format((estaciones.sel == 1).sum()))

## Exportar resultados

In [ ]:
# exportar todas las estaciones juntas
estaciones.to_file(cfg.path_GIS / 'estaciones.shp', driver='ESRI Shapefile', index=True)

# exportar las series de caudal
caudal.to_parquet(cfg.path_out / 'caudal.parquet')

# exportar todos los embalses juntos
embalses.rename(columns={'nom_embalse': 'nombre'}, inplace=True)
embalses.to_file(cfg.path_GIS / 'embalses.shp', driver='ESRI Shapefile', index=True)

In [ ]:
caudal.shape

In [ ]:
caudal.head()